# 调用、SOC与履约损耗 · v2.4

**研究问题**：调用能量、SOC恢复和履约损耗怎样改变贡献？

**数据口径**：官方JEPX/EPRX/NESO原始快照与具出处的制度事实A；归一化/派生计算B；报价、中标率、调用、流动性、可靠性、成本和DCF为C。

**主要结果**：由下列单独模块从原始输入重新计算，在末单元才与正式输出比较。

**最大限制**：无日本项目逐笔投标、高频调用、真实日内成交或Taoke运行/财务日志；这些缺口不以合成数据冒充。

本Notebook依赖Core中的原始来源、声明假设与Python模块。历史验收ZIP不属于新七个Notebook的输入。

In [1]:
MODULE = 'activation'
from pathlib import Path
import sys, json, gzip, hashlib, tempfile
sys.dont_write_bytecode = True
root = Path.cwd().resolve()
while not (root / 'scripts/operations/common.py').is_file():
    if root == root.parent:
        raise FileNotFoundError('请在验收包的 Japan BESS 项目目录中运行此 Notebook')
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))
from operations.common import Battery, read_json, qstats
from operations import data, bidding, activation, joint, portfolio, reliability, markets, updater
battery = Battery()
replay = root / 'tmp/notebook-replays' / MODULE
replay.mkdir(parents=True, exist_ok=True)
print('Model:', MODULE, 'Battery:', battery.power_mw, 'MW /', battery.energy_mwh, 'MWh')


Model: activation Battery: 2.0 MW / 8.14 MWh


## 原始来源到规范化

核对SHA、主键、半小时/UTC时间、单位与完整交割日；不插值，不把无成交价格改为零。

In [2]:
normalized = data.normalize_all(replay/'normalized')
assert normalized['quality']['status'] == 'passed'
print(json.dumps(normalized['quality'], ensure_ascii=False, indent=2))

{
  "status": "passed",
  "price_days": 643,
  "study_days": 188,
  "eprx_rows": 90240,
  "neso_rows": 60924,
  "duplicate_keys": 0,
  "missing_study_partitions": 0,
  "imputation": "none",
  "source_hashes_verified": true,
  "legacy_prices_exact_match": true,
  "eprx_gate": "B",
  "historical_eprx_available_at": "null; descriptive only"
}


## 声明假设与独立计算

所有参数按A/B/C分列。完整算法在各自Python模块，不把网站展示数值硬编码成模型结果。

In [3]:
assumptions = read_json(root/'research/operations-v2-4/assumptions.json')
print(json.dumps(assumptions['physical'],ensure_ascii=False,indent=2))

{
  "power_mw": 2,
  "energy_mwh": 8.14,
  "rte": 0.88,
  "min_soc": 0.1,
  "max_soc": 0.9,
  "initial_and_terminal_soc": 0.5,
  "max_daily_efc": 1,
  "shadow_wear_jpy_kwh_internal_throughput": 2,
  "auxiliary_mw": 0.005
}


In [4]:
result = activation.analyze(normalized, battery)

## 物理、数学和信息可用性复核

边界断言和独立公式先于正式结果比对。更完整的golden与扰动测试保存在tests/operations。

In [5]:
from operations.common import soc_step
for case in result['parameter_cases']:
    energy = case['initial_soc_pct']/100*battery.energy_mwh
    for row in case['timeline']:
        energy = soc_step(energy, row['grid_net_mw'], battery)
        assert abs(energy-row['soc_mwh']) < 1e-7
        assert battery.e_min-1e-8 <= energy <= battery.e_max+1e-8
assert not result['qualification']['qualification_proven_by_energy_model']
print([(x['scenario'], round(x['efc'],3), round(x['net_vs_spot_opportunity_jpy'])) for x in result['cases']])


[('Low', 0.018, 59457), ('Base', 0.074, 40466), ('High', 0.235, -5537)]


In [6]:
# 计算完成之后才读取正式结果作一致性比较；正式结果不作为模型输入。
result['scope'] = result['scope'].replace('188', str(normalized['quality']['study_days']))
if MODULE == 'portfolio':
    result['correlations']['observations'] = normalized['quality']['study_days']
canonical = json.loads(gzip.decompress((root/'research/operations-v2-4/results'/f'{MODULE}.json.gz').read_bytes()))
def semantic_hash(obj):
    return hashlib.sha256(json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
fresh_sha = semantic_hash(result)
expected_sha = semantic_hash(canonical)
assert fresh_sha == expected_sha, 'fresh result differs from reviewed calculation; investigate before sharing'
(replay/'fresh-result.json.gz').write_bytes(gzip.compress(json.dumps(result,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode(),mtime=0))
print('PASS: raw hashes -> fresh normalization -> fresh model; semantic SHA-256',fresh_sha)
print('No saved model cache was used as input.')


PASS: raw hashes -> fresh normalization -> fresh model; semantic SHA-256 fd8cb32f8b0359ac250a1916c3c515658ae006bdda122c630589802cc442153f
No saved model cache was used as input.


## 结果边界与出处

来源登记：research/operations-v2-4/source-registry.json。正式参数：assumptions.json。时间顺序：market-timeline.json。独立报告：japan-bess-operations-research-v2-4.pdf，页7, 8, 9。

样本年化不是全年实收；影子损耗和换电现金不重复扣减；公开规格组合不是实际统一运营资产池。